# Save/Convert/Load MERA-Files
The RAMSES simulation data is stored in JLD2 file format and can be accessed from these files. Our high-resolution galaxy simulations, run on over 5,000 cores, show that using compressed Mera files greatly decreases storage requirements and accelerates data loading compared to standard RAMSES files. Refer to the Benchmarks section.

## Quick Reference

### Essential Functions
```julia
# Convert from RAMSES files multiple data to JLD2
convertdata(output_num, path="ramses_path", fpath="jld2_path")
convertdata(output_num, [:hydro, :particles], path="ramses_path", fpath="jld2_path")

# Save individual loaded datasets
savedata(data_object, "output_path", fmode=:write)   # Create new file
savedata(data_object, "output_path", fmode=:append)  # Add to existing file

# Load from JLD2
loaddata(output_num, "jld2_path", :hydro)
loaddata(output_num, "jld2_path", :particles) 
loaddata(output_num, "jld2_path", :gravity)

# Load with spatial selection
loaddata(output_num, "jld2_path", :hydro, 
         xrange=[-10,10], yrange=[-10,10], zrange=[-2,2], 
         center=[:boxcenter], range_unit=:kpc)

# View and inspect stored data
viewdata(output_num, "jld2_path")                    # Show file contents
infodata(output_num, "jld2_path", :hydro)           # Data type info
```

### Key File Modes
- `:write` - Create new file or overwrite existing (use for first save)
- `:append` - Add data types to existing file (safe for additional data)

### Data Types
- `:hydro` - Gas density, velocity, pressure, temperature
- `:particles` - Stellar/DM particles: position, velocity, mass, age  
- `:gravity` - Gravitational potential and force fields
- `:clumps` - Structure identification data
- `:rt` - Radiative-transfer photon densities and fluxes
- `:sinks` - Sink particles: accreting point masses (see
  [Sink Data: First Inspection](01_sinks_First_Inspection.md))

In [1]:
using Mera


*__   __ _______ ______   _______ 


|  |_|  |       |    _ | |   _   |
|       |    ___|   | || |  |_|  |
|       |   |___|   |_||_|       |
|       |    ___|    __  |       |
| ||_|| |   |___|   |  | |   _   |
|_|   |_|_______|___|  |_|__| |__|
Mera v1.8.0 | Julia 1.12.7 | 4 threads



## Load the Data From Ramses

In [2]:
# Example-data root. Point this at your own simulation folder, or set the
# MERA_EXAMPLES environment variable; every path below is built from it.
MERA_EXAMPLES = get(ENV, "MERA_EXAMPLES", "/Volumes/FASTStorage/Simulations/Mera-Tests");

info = getinfo(300,  "$MERA_EXAMPLES/RAMSES/mw_L10");
gas  = gethydro(info, verbose=false, show_progress=false); 
part = getparticles(info, verbose=false, show_progress=false); 
grav = getgravity(info, verbose=false, show_progress=false); 
# the same applies for clump-data...

[Mera]: 2026-08-31T13:49:45.763



Code: RAMSES
output [300] summary:
mtime: 

2023-04-09T05:34:09
ctime: 2025-06-21T18:31:24.020
simulation time: 445.89 [Myr]
boxlen: 48.0 [kpc]
ncpu: 640
ndim: 3
cosmological:  false
-------------------------------------------------------
amr:           true
level(s): 6 - 10 --> cellsize(s): 750.0 [pc] - 46.88 [pc]
-------------------------------------------------------
hydro:         true
hydro-variables:  

7  --> (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
hydro-descriptor: (:density, :velocity_x, :velocity_y, :velocity_z, :pressure, :scalar_00, :scalar_01)
γ: 1.6667
-------------------------------------------------------
gravity:       true
gravity-variables: (:epot, :ax, :ay, :az)
-------------------------------------------------------
particles:     true
- Nstars:   5.445150e+05 
particle-variables: 

7  --> (:vx, :vy, :vz, :mass, :family, :tag, :birth)
particle-descriptor: (:position_x, :position_y, :position_z, :velocity_x, :velocity_y, :velocity_z, :mass, :identity, :levelp, :family, :tag, :birth_time)
-------------------------------------------------------
rt:            false
clumps:           false
-------------------------------------------------------
namelist-file: ("&COOLING_PARAMS", "&SF_PARAMS", "&AMR_PARAMS", "&BOUNDARY_PARAMS", "&OUTPUT_PARAMS", "&POISSON_PARAMS", "&RUN_PARAMS", "

&FEEDBACK_PARAMS", "&HYDRO_PARAMS", "&INIT_PARAMS", "&REFINE_PARAMS")
-------------------------------------------------------
timer-file:       true
compilation-file: false
makefile:         true
patchfile:        true



## Store the Data Into JLD2 Files
The running number is taken from the original RAMSES outputs.

In [3]:
savedata(gas, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/");

[Mera]: 2026-08-31T13:51:07.698


Not existing file: output_00300.jld2
Directory: /Volumes/FASTStorage/Simulations/Mera-Tests/RAMSES/mw_L10
-----------------------------------
merafile_version: 1.0  -  Simulation code: RAMSES
-----------------------------------
DataType: hydro

  -  Data variables: (:level, :cx, :cy, :cz, :rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
-----------------------------------
I/O mode: nothing  -  Compression: false
-----------------------------------
-----------------------------------
Memory size: 

2.321 GB (uncompressed)
-----------------------------------



<div class="alert alert-block alert-info"> <b>NOTE</b> The hydro data was not written into the file to prevent overwriting existing files.

The following argument is mandatory: **fmode=:write** </div>

In [4]:
savedata(gas, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", fmode=:write);

[Mera]: 2026-08-31T13:51:09.054




Create file: output_00300.jld2
Directory: /Volumes/FASTStorage/Simulations/Mera-Tests/RAMSES/mw_L10
-----------------------------------
merafile_version: 1.0  -  Simulation code: RAMSES
-----------------------------------
DataType: hydro  -  Data variables: (:level, :cx, :cy, :cz, :rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
-----------------------------------
I/O mode: write

  -  Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------


JLD2  0.6.5
CodecBzip2  0.8.5
CodecZlib  0.7.8
CodecLz4  0.4.6
Mera  1.8.0
-----------------------------------


Memory size: 2.321 GB (uncompressed)
Total file size: 1.275 GB
-----------------------------------



Add/Append further datatypes:

In [5]:
savedata(part, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", fmode=:append);
savedata(grav, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", fmode=:append);

[Mera]: 2026-08-31T13:51:17.436


Create file: output_00300.jld2
Directory: /Volumes/FASTStorage/Simulations/Mera-Tests/RAMSES/mw_L10
-----------------------------------
merafile_version: 1.0  -  Simulation code: RAMSES
-----------------------------------
DataType: particles  -  Data variables: (:level, :x, :y, :z, :id, :family, :tag, :vx, :vy, :vz, :mass, :birth)
-----------------------------------
I/O mode: append  -  Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------


JLD2  0.6.5
CodecBzip2  0.8.5
CodecZlib  0.7.8
CodecLz4  0.4.6
Mera  1.8.0
-----------------------------------


Memory size: 38.449 MB (uncompressed)
Total file size: 1.306 GB
-----------------------------------

[Mera]: 2026-08-31T13:51:18.653


Create file: output_00300.jld2
Directory: /Volumes/FASTStorage/Simulations/Mera-Tests/RAMSES/mw_L10
-----------------------------------
merafile_version: 1.0  -  Simulation code: RAMSES
-----------------------------------
DataType: gravity  -  Data variables: (:level, :cx, :cy, :cz, :epot, :ax, :ay, :az)
-----------------------------------
I/O mode: append  -  Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------


JLD2  0.6.5
CodecBzip2  0.8.5
CodecZlib  0.7.8
CodecLz4  0.4.6
Mera  1.8.0
-----------------------------------


Memory size: 1.688 GB (uncompressed)
Total file size: 2.158 GB
-----------------------------------



<div class="alert alert-block alert-info"> <b>NOTE</b> It is not possible to exchange stored data; only writing into a new file or appending is supported. </div>

## Overview of Stored Data

In [6]:
vd = viewdata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/")

[Mera]: 2026-08-31T13:51:23.101



Mera-file output_00300.jld2 contains:

Datatype: 

particles
merafile_version: 1.0
Compression: JLD2Lz4.Lz4Filter(0x40000000)
CodecZlib: 

VersionNumber[v"0.7.8"]
merafile_version: 1.0
JLD2: VersionNumber[v"0.6.5"]
CodecBzip2: VersionNumber[v"0.8.5"]
JLD2compatible_versions: (lower = v"0.1.0", upper = v"0.3.0")
CodecLz4: VersionNumber[v"0.4.6"]
Mera: VersionNumber[v"1.8.0"]
-------------------------
Memory: 38.449326515197754 MB (uncompressed)


Datatype: gravity
merafile_version: 1.0
Compression: JLD2Lz4.Lz4Filter(0x40000000)
CodecZlib: VersionNumber[v"0.7.8"]
merafile_version: 1.0
JLD2: VersionNumber[v"0.6.5"]
CodecBzip2: VersionNumber[v"0.8.5"]
JLD2compatible_versions: (lower = v"0.1.0", upper = v"0.3.0")
CodecLz4: VersionNumber[v"0.4.6"]
Mera: VersionNumber[v"1.8.0"]
-------------------------
Memory: 1.6880828058347106 GB (uncompressed)


Datatype: hydro
merafile_version: 1.0
Compression: JLD2Lz4.Lz4Filter(0x40000000)
CodecZlib: VersionNumber[v"0.7.8"]
merafile_version: 1.0
JLD2: VersionNumber[v"0.6.5"]
CodecBzip2: VersionNumber[v"0.8.5"]
JLD2compatible_versions: (lower = v"0.1.0", upper = v"0.3.0")
CodecLz4: VersionN

Dict{Any, Any} with 4 entries:
  "particles" => Dict{Any, Any}("versions"=>Dict{Any, Any}("CodecZlib"=>Version…
  "FileSize"  => (2.158, "GB")
  "gravity"   => Dict{Any, Any}("versions"=>Dict{Any, Any}("CodecZlib"=>Version…
  "hydro"     => Dict{Any, Any}("versions"=>Dict{Any, Any}("CodecZlib"=>Version…

Information about the content, etc. is returned in a dictionary.

Get a detailed tree-view of the data-file:

In [7]:
vd = viewdata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", showfull=true)

[Mera]: 2026-08-31T13:51:23.781

Mera-file output_00300.jld2 contains:

 ├─📂 hydro
 │  ├─🔢 data
 │  ├─🔢 info
 │  └─📂 information
 │     ├─🔢 compression
 │     ├─🔢 comments
 │     ├─🔢 storage
 │     ├─🔢 memory
 │     └─📂 versions
 │        ├─🔢 merafile_version
 │        ├─🔢 JLD2compatible_versions
 │        ├─🔢 JLD2
 │        ├─🔢 CodecBzip2
 │        ├─🔢 CodecZlib
 │        ├─🔢 CodecLz4
 │        └─🔢 Mera
 ├─📂 particles
 │  ├─🔢 data
 │  ├─🔢 info
 │  └─📂 information
 │     ├─🔢 compression
 │     ├─🔢 comments
 │     ├─🔢 storage
 │     ├─🔢 memory
 │     └─📂 versions
 │        ├─🔢 merafile_version
 │        ├─🔢 JLD2compatible_versions
 │        ├─🔢 JLD2
 │        ├─🔢 CodecBzip2
 │        ├─🔢 CodecZlib
 │        ├─🔢 CodecLz4
 │        └─🔢 Mera
 └─📂 gravity
    ├─🔢 data
    ├─🔢 info
    └─📂 information
       ├─🔢 compression
       ├─🔢 comments
       ├─🔢 storage
       ├─🔢 memory
       └─📂 versions
          ├─🔢 merafile_version
          ├─🔢 JLD2compatible_versions
          ├─🔢 JLD2
     

Dict{Any, Any} with 4 entries:
  "particles" => Dict{Any, Any}("versions"=>Dict{Any, Any}("CodecZlib"=>Version…
  "FileSize"  => (2.158, "GB")
  "gravity"   => Dict{Any, Any}("versions"=>Dict{Any, Any}("CodecZlib"=>Version…
  "hydro"     => Dict{Any, Any}("versions"=>Dict{Any, Any}("CodecZlib"=>Version…

## Get Info
The following function **infodata** is comparable to **getinfo()** used for the RAMSES files and loads detailed information about the simulation output:

In [8]:
info = infodata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/");

[Mera]: 2026-08-31T13:51:23.949



Use datatype: hydro
Code: 

RAMSES
output [300] summary:
mtime: 2023-04-09T05:34:09
ctime: 2025-06-21T18:31:24.020
simulation time: 445.89 [Myr]
boxlen: 48.0 [kpc]
ncpu: 640
ndim: 3
cosmological:  false
-------------------------------------------------------
amr:           true
level(s): 6 - 10 --> cellsize(s): 750.0 [pc] - 46.88 [pc]
-------------------------------------------------------
hydro:         true
hydro-variables:  7  --> (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
hydro-descriptor: (:density, :velocity_x, :velocity_y, :velocity_z, :pressure, :scalar_00, :scalar_01)
γ: 1.6667
-------------------------------------------------------
gravity:       true
gravity-variables: (:epot, :ax, :ay, :az)
-------------------------------------------------------
particles:     true
- Nstars:   5.445150e+05 
particle-variables: 7  --> (:vx, :vy, :vz, :mass, :family, :tag, :birth)
particle-descriptor: (:position_x, :position_y, :position_z, :velocity_x, :velocity_y, :velocity_z, :mass, :identity, :levelp, :family

In this case, it loaded the **InfoDataType** from the **hydro** data. Choose a different stored **datatype** to get the info from:

In [9]:
info = infodata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", :particles);

[Mera]: 2026-08-31T13:51:24.606

Use datatype: particles
Code: RAMSES
output [300] summary:
mtime: 2023-04-09T05:34:09
ctime: 2025-06-21T18:31:24.020
simulation time: 445.89 [Myr]
boxlen: 48.0 [kpc]
ncpu: 640
ndim: 3
cosmological:  false
-------------------------------------------------------
amr:           true
level(s): 6 - 10 --> cellsize(s): 750.0 [pc] - 46.88 [pc]
-------------------------------------------------------
hydro:         true
hydro-variables:  7  --> (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
hydro-descriptor: (:density, :velocity_x, :velocity_y, :velocity_z, :pressure, :scalar_00, :scalar_01)
γ: 1.6667
-------------------------------------------------------
gravity:       true
gravity-variables: (:epot, :ax, :ay, :az)
-------------------------------------------------------
particles:     true
- Nstars:   5.445150e+05 
particle-variables: 7  --> (:vx, :vy, :vz, :mass, :family, :tag, :birth)
particle-descriptor: (:position_x, :position_y, :position_z, :velocity_

## Load The Data from JLD2

### Full Data

In [10]:
gas = loaddata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", :hydro);

[Mera]: 2026-08-31T13:51:24.707



Open Mera-file output_00300.jld2:

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Memory used for data table :2.321106480434537

 GB
-------------------------------------------------------



In [11]:
typeof(gas)

HydroDataType

In [12]:
part = loaddata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", :particles);

[Mera]: 2026-08-31T13:51:26.227

Open Mera-file output_00300.jld2:

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Memory used for data table :38.44944095611572

 MB
-------------------------------------------------------



In [13]:
typeof(part)

PartDataType

### Data Range
Complete data is loaded, and the selected subregion is returned:

In [14]:
gas = loaddata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", :hydro,
                    xrange=[-10,10], 
                    yrange=[-10,10], zrange=[-2,2],
                    center=[:boxcenter], 
                    range_unit=:kpc);

[Mera]: 2026-08-31T13:51:26.563

Open Mera-file output_00300.jld2:

center: [0.5, 0.5, 0.5] 

==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.2916667 :: 0.7083333  	==> 14.0 [kpc] :: 34.0 [kpc]
ymin::ymax: 0.2916667 :: 0.7083333  	==> 14.0 [kpc] :: 34.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

Memory used for data table :

580.297945022583 MB
-------------------------------------------------------



## Convert RAMSES Output Into JLD2
Existing AMR, hydro, gravity, particle, and clump data is sequentially stored in a JLD2 file. The individual loading/writing processes are timed, and the memory usage is returned in a dictionary:

### Full Data

In [15]:
cvd = convertdata(300, path="$MERA_EXAMPLES/RAMSES/mw_L10",
                  fpath="$MERA_EXAMPLES/MERA-FILES/JLD2_files/");

[Mera]: 2026-08-31T13:51:28.685



Requested datatypes: [:hydro, :gravity, :particles, :clumps, :rt, :sinks]
Max threads: 4 of 4 available
Threading applied to: hydro, gravity, particles
Threading NOT applied to: clumps (single-threaded by design)

domain:


xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]


reading/writing lmax: 10 of 10
-----------------------------------
Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------
- hydro (threaded: max_threads=4)


Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   1%|▋                                                 |  ETA: 0:00:40 (63.00 ms/it)

...

Processing files:  99%|█████████████████████████████████████████████████▉|  ETA: 0:00:00 (21.60 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:13 (21.60 ms/it)



✓ File processing complete! Combining results...


- particles (threaded: max_threads=4)



Final Statistics:
- total folder size: 5.682 GB
- selected data size: 5.68 GB
- peak memory used: 4.047 GB
- compressed file size: 2.158 GB
- compression ratio: 0.38
- data reduction: 62.0%
- total processing time: 88.43 seconds
- effective threads: 4


#### Timer
Get a view of the timers:

In [16]:
using Mera.TimerOutputs

In [17]:
cvd

Dict{Any, Any} with 5 entries:
  "threading"    => Dict{Any, Any}("max_threads_requested"=>4, "julia_version"=…
  "viewdata"     => Dict{Any, Any}("particles"=>Dict{Any, Any}("versions"=>Dict…
  "size"         => Dict{Any, Any}("folder"=>Any[6101111412, "Bytes"], "selecte…
  "benchmark"    => Dict{Any, Any}("xrange"=>[missing, missing], "subset"=>fals…
  "TimerOutputs" => Dict{Any, Any}("writing"=>─────────────────────────────────…

In [18]:
cvd["TimerOutputs"]["reading"]

──────────────────────────────────────────────────────────────────────
                             Time                    Allocations      
                    ───────────────────────   ────────────────────────
 Tot / % measured:       88.9s /  85.0%            100GiB /  95.1%    

Section     ncalls     time    %tot     avg     alloc    %tot      avg
──────────────────────────────────────────────────────────────────────
hydro            1    57.1s   75.6%   57.1s   76.1GiB   79.8%  76.1GiB
gravity          1    17.0s   22.5%   17.0s   17.6GiB   18.4%  17.6GiB
particles        1    1.39s    1.8%   1.39s   1.71GiB    1.8%  1.71GiB
──────────────────────────────────────────────────────────────────────

In [19]:
cvd["TimerOutputs"]["writing"]

──────────────────────────────────────────────────────────────────────
                             Time                    Allocations      
                    ───────────────────────   ────────────────────────
 Tot / % measured:       88.9s /  14.0%            100GiB /   4.9%    

Section     ncalls     time    %tot     avg     alloc    %tot      avg
──────────────────────────────────────────────────────────────────────
gravity          1    7.11s   57.2%   7.11s   1.97GiB   40.5%  1.97GiB
hydro            1    4.84s   38.9%   4.84s   2.85GiB   58.5%  2.85GiB
particles        1    477ms    3.8%   477ms   52.3MiB    1.0%  52.3MiB
──────────────────────────────────────────────────────────────────────

In [20]:
# prep timer
to = TimerOutput();

In [21]:
@timeit to "MERA" begin
    @timeit to "hydro"     gas = loaddata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", :hydro, )
    @timeit to "particles" part= loaddata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", :particles)
end;

[Mera]: 2026-08-31T13:52:58.128



Open Mera-file output_00300.jld2:

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Memory used for data table :2.321106480434537

 GB
-------------------------------------------------------

[Mera]: 2026-08-31T13:53:03.589

Open Mera-file output_00300.jld2:

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Memory used for data table :38.44944095611572

 MB
-------------------------------------------------------



In [22]:
to

────────────────────────────────────────────────────────────────────────
                               Time                    Allocations      
                      ───────────────────────   ────────────────────────
  Tot / % measured:        6.04s /  92.3%           5.30GiB /  98.9%    

Section       ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────────
MERA               1    5.58s  100.0%   5.58s   5.24GiB  100.0%  5.24GiB
  hydro            1    5.46s   97.9%   5.46s   5.16GiB   98.5%  5.16GiB
  particles        1    115ms    2.1%   115ms   81.0MiB    1.5%  81.0MiB
────────────────────────────────────────────────────────────────────────

<div class="alert alert-block alert-info"> <b>NOTE</b> The reading from JLD2 files is multiple times faster than from the original RAMSES files. </div>

#### Used Memory

In [23]:
cvd["size"]

Dict{Any, Any} with 4 entries:
  "folder"   => Any[6101111412, "Bytes"]
  "selected" => Any[6.09885e9, "Bytes"]
  "ondisc"   => Any[2317448537, "Bytes"]
  "used"     => Any[4.34515e9, "Bytes"]

<div class="alert alert-block alert-info"> <b>NOTE</b> The compressed JLD2 file takes a significantly smaller disk space than the original RAMSES folder.</div>

In [24]:
factor = cvd["size"]["folder"][1] / cvd["size"]["ondisc"][1]
println("==============================================================================")
println("In this example, the disk space is reduced by a factor of $factor !!")
println("==============================================================================")

In this example, the disk space is reduced by a factor of 2.632684745568527 !!


### Selected Datatypes

In [25]:
cvd = convertdata(300, [:hydro, :particles], 
                  path="$MERA_EXAMPLES/RAMSES/mw_L10",
                  fpath="$MERA_EXAMPLES/MERA-FILES/JLD2_files/");

[Mera]: 2026-08-31T13:53:03.969



Requested datatypes: [:hydro, :particles]
Max threads: 4 of 4 available
Threading applied to: hydro, gravity, particles
Threading NOT applied to: clumps (single-threaded by design)

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]


reading/writing lmax: 10 of 10
-----------------------------------
Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------
- hydro (threaded: max_threads=4)


Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   0%|▎                                                 |  ETA: 0:01:13 ( 0.11  s/it)

Processing files:   2%|█                                                 |  ETA: 0:00:42 (66.42 ms/it)

...

Processing files:  99%|█████████████████████████████████████████████████▉|  ETA: 0:00:00 (41.41 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:26 (41.36 ms/it)



✓ File processing complete! Combining results...


- particles (threaded: max_threads=4)



Final Statistics:
- total folder size: 5.682 GB
- selected data size: 4.002 GB
- peak memory used: 2.359 GB
- compressed file size: 1.306 GB
- compression ratio: 0.326
- data reduction: 77.0%
- total processing time: 75.32 seconds
- effective threads: 4


### What survives the round trip

A mera file stores the data **table**, not a reduced summary, so every column comes back exactly as
it was read from RAMSES, including the ones that identify *what* a particle is. `:family` and
`:tag` are ordinary columns and are written and read back unchanged, so particle-type selection
works identically on a mera file and on the original output:

```julia
parts = loaddata(3, "path/to/mera_files", :particles)
getparticlemask(parts, :tracer)      # same result as on the RAMSES output
getmask(parts, :family, ==(2.0))     # :family behaves like any other quantity
```

This is worth knowing before converting a large run: the conversion is lossless for selection
purposes, so you do not have to keep the original output around in order to separate dark matter
from stars, or tracers from either. The same applies to the values themselves, a round trip
reproduces the RAMSES read bit-for-bit, which is one of the properties Mera's own test suite
asserts on every snapshot of a public fixture.

## Compression

Mera files are LZ4-compressed. This build runs on JLD2 0.6, whose compression is provided by
JLD2Lz4, **LZ4 is the only codec available**. Passing `compress=true` (or leaving it out)
gives you LZ4; passing `compress=false` writes uncompressed.

!!! note "Other compressors are accepted but substituted"
    For backwards compatibility, `ZlibCompressor` and `Bzip2Compressor` are still accepted as
    arguments, but Mera warns and writes LZ4 instead, so do not choose one expecting a
    smaller file. The cells below demonstrate exactly that: watch for the warning in the
    output.


| Argument | What you get |
|---|---|
| `compress=true` (or omitted) | LZ4, the default |
| `compress=false` | no compression |
| `ZlibCompressor()` / `Bzip2Compressor()` | accepted, but **substituted with LZ4** and a warning |
| any other JLD2-accepted filter | passed through to JLD2 unchanged |


Passing a compressor Mera cannot use shows what happens, the call succeeds, the file is
written, and the warning tells you the codec was swapped:


In [26]:
using Mera.CodecZlib
cvd = convertdata(300, [:hydro, :particles], compress=ZlibCompressor(),
                  path="$MERA_EXAMPLES/RAMSES/mw_L10",
                  fpath="$MERA_EXAMPLES/MERA-FILES/JLD2_files/");

[Mera]: 2026-08-31T13:54:19.669



Requested datatypes: [:hydro, :particles]
Max threads: 4 of 4 available
Threading applied to: hydro, gravity, particles
Threading NOT applied to: clumps (single-threaded by design)

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]



┌ Warning: This Mera build (JLD2 0.6) supports LZ4 compression only — using LZ4 instead of ZlibCompressor.
└ @ Mera ~/code-github/Mera.jl/src/functions/data/data_save.jl:281



reading/writing lmax: 10 of 10
-----------------------------------
Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------
- hydro (threaded: max_threads=4)


Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

...

Processing files:  99%|█████████████████████████████████████████████████▌|  ETA: 0:00:00 (40.49 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:25 (40.50 ms/it)



✓ File processing complete! Combining results...


- particles (threaded: max_threads=4)



Final Statistics:
- total folder size: 5.682 GB
- selected data size: 4.002 GB
- peak memory used: 2.359 GB
- compressed file size: 1.306 GB
- compression ratio: 0.326
- data reduction: 77.0%
- total processing time: 71.66 seconds
- effective threads: 4


In [27]:
savedata(gas, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", 
            fmode=:write, compress=ZlibCompressor());

[Mera]: 2026-08-31T13:55:31.362


Create file: output_00300.jld2
Directory: /Volumes/FASTStorage/Simulations/Mera-Tests/RAMSES/mw_L10
-----------------------------------
merafile_version: 1.0  -  Simulation code: RAMSES
-----------------------------------
DataType: hydro  -  Data variables: (:level, :cx, :cy, :cz, :rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
-----------------------------------
I/O mode: write  -  Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------
JLD2  0.6.5
CodecBzip2  0.8.5
CodecZlib  0.7.8
CodecLz4  0.4.6
Mera  1.8.0
-----------------------------------


Memory size: 2.321 GB (uncompressed)
Total file size: 1.275 GB
-----------------------------------



Get more information about the parameters of the compressor:

In [28]:
?ZlibCompressor

search: 

ZlibCompressor ZlibDecompressor GzipCompressor ZlibCompressorStream



```julia
ZlibCompressor(;level=-1, windowbits=15)
```

Create a zlib compression codec.

## Arguments

  * `level` (-1..9): compression level. 1 gives best speed, 9 gives best compression, 0 gives no compression at all (the input data is simply copied a block at a time). -1 requests a default compromise between speed and compression (currently equivalent to level 6).
  * `windowbits` (9..15): size of history buffer is `2^windowbits`.

!!! warning
    `serialize` and `deepcopy` will not work with this codec due to stored raw pointers.



## Comments
Add a description to the files:

In [29]:
comment = "The simulation is...."
cvd = convertdata(300, [:hydro, :particles], comments=comment,
                  path="$MERA_EXAMPLES/RAMSES/mw_L10",
                  fpath="$MERA_EXAMPLES/MERA-FILES/JLD2_files/");

[Mera]: 2026-08-31T13:55:39.753



Requested datatypes: [:hydro, :particles]
Max threads: 4 of 4 available
Threading applied to: hydro, gravity, particles
Threading NOT applied to: clumps (single-threaded by design)

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]


reading/writing lmax: 10 of 10
-----------------------------------
Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------
- hydro (threaded: max_threads=4)


Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   0%|▎                                                 |  ETA: 0:00:50 (79.27 ms/it)

Processing files:   2%|█                                                 |  ETA: 0:00:28 (44.82 ms/it)

...

Processing files:  99%|█████████████████████████████████████████████████▉|  ETA: 0:00:00 (27.28 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:17 (27.23 ms/it)



✓ File processing complete! Combining results...


- particles (threaded: max_threads=4)



Final Statistics:
- total folder size: 5.682 GB
- selected data size: 4.002 GB
- peak memory used: 2.359 GB
- compressed file size: 1.306 GB
- compression ratio: 0.326
- data reduction: 77.0%
- total processing time: 68.01 seconds
- effective threads: 4


In [30]:
comment = "The simulation is...."
savedata(gas, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", comments=comment, fmode=:write);

[Mera]: 2026-08-31T13:56:47.797


Create file: output_00300.jld2
Directory: /Volumes/FASTStorage/Simulations/Mera-Tests/RAMSES/mw_L10
-----------------------------------
merafile_version: 1.0  -  Simulation code: RAMSES
-----------------------------------
DataType: hydro  -  Data variables: (:level, :cx, :cy, :cz, :rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
-----------------------------------
I/O mode: write  -  Compression: JLD2Lz4.Lz4Filter(0x40000000)
-----------------------------------
JLD2  0.6.5
CodecBzip2  0.8.5
CodecZlib  0.7.8
CodecLz4  0.4.6
Mera  1.8.0
-----------------------------------


Memory size: 2.321 GB (uncompressed)
Total file size: 1.275 GB
-----------------------------------



Load the comment (hydro) from JLD2 file:

In [31]:
vd = viewdata(300, "$MERA_EXAMPLES/MERA-FILES/JLD2_files/", verbose=false);

In [32]:
vd["hydro"]["comments"]

"The simulation is...."